# Week 10 guided practice

CML3113. Follow the lecturer’s PowerPoint in this order. This notebook contains worked examples, not your graded upload.

Extract the student ZIP and upload engineering_data.csv into Colab with the Files panel before running the cells. The target in this notebook is crusher energy demand in kWh/t.

## Today’s learning

*PowerPoint slide 2*

WHAT WE WILL DO
Watch a model memorise its training rows, learn to check it on several held-back groups, and find four ways a score can leak. Learn the words overfitting, cross-validation and data leakage.

HOW YOU WILL SHOW IT
You can explain why training error cannot be trusted, and how you would tell that a score is leaking. You can split by equipment group and report an error with its spread.

## The supplied file

*PowerPoint slide 4*

Extract this week’s student ZIP. Upload engineering_data.csv into Colab using the Files panel, or keep them beside the notebook locally.

The examples below read this file. Run the cells in slide order. Keep the original CSV unchanged.

## Loading the supplied observations

*PowerPoint slide 5*

raw_data: original CSV table; keep it unchanged so the analysis can be checked.

Run the cell and read the output below it.

In [ ]:
import pandas as pd
raw_data = pd.read_csv("engineering_data.csv")
print(raw_data.head())
print(raw_data.shape)
print(raw_data.dtypes)

## Three sets have different jobs

*PowerPoint slide 6*

Which set must stay sealed until choices are fixed?

## Rows used for modelling

*PowerPoint slide 7*

Keep the first occurrence of each record_id. Retain recorded targets and load_pct within 0–100. Then separate E1–E4 for training, E5 for validation and E6 for the final test_data.

Fill missing input values using training data only. Keep final-test targets out of model selection.

## Preparing the supplied modelling rows: step 1

*PowerPoint slide 8*

modelling_data: copied observations after duplicate and validity checks.

Run the cell and read the output below it.

In [ ]:
modelling_data = raw_data.drop_duplicates("record_id").copy()
target_is_recorded = modelling_data["target"].notna()
load_is_valid = modelling_data["load_pct"].between(0, 100)
rows_to_keep = target_is_recorded & load_is_valid
print(target_is_recorded.head())
print(load_is_valid.head())
modelling_data = modelling_data[rows_to_keep].copy()
print(len(raw_data), len(modelling_data))

## Choose groups before choosing rows

*PowerPoint slide 9*

isin(list_of_labels) answers True when a row’s equipment ID appears in the list.

Here E1–E4 supply training rows, E5 validation rows and E6 test rows.

The groups stay separate so one equipment group cannot appear on both sides.

## Preparing the supplied modelling rows: step 2

*PowerPoint slide 10*

modelling_data: copied observations after duplicate and validity checks.

Run the cell and read the output below it.

In [ ]:
is_training_equipment = modelling_data["equipment_id"].isin(
    ["E1", "E2", "E3", "E4"]
)
is_validation_equipment = modelling_data["equipment_id"].isin(
    ["E5"]
)
is_test_equipment = modelling_data["equipment_id"].isin(["E6"])

## Preparing the supplied modelling rows: step 3

*PowerPoint slide 11*

modelling_data: copied observations after duplicate and validity checks.

Run the cell and read the output below it.

In [ ]:
training_data = modelling_data[is_training_equipment].copy()
validation_data = modelling_data[is_validation_equipment].copy()
test_data = modelling_data[is_test_equipment].copy()
feature_columns = ["load_pct", "ambient_c"]
print(len(training_data), len(validation_data), len(test_data))

## Training-only filling

*PowerPoint slide 12*

median ignores missing values. Calculate it from training input values and reuse it for validation gaps.

Do not calculate a new median from validation or test observations. A pipeline keeps this rule with the model.

## Operations for the next example

*PowerPoint slide 13*

01

Middle recorded value: series.median() sorts known values conceptually and returns the middle one. With an even count, it averages the two middle values. Missing entries do not enter the calculation.

02

Replacing missing entries: series.fillna(value) returns a column whose missing entries contain value. Recorded values stay unchanged. Assign the returned column to retain the fill.

03

Rounded display: series.round(3) returns values rounded to three decimal places. Python round(number, 3) rounds one number. Keep unrounded calculations for returned assessment answers.

## Operations for the next example

*PowerPoint slide 14*

Adding values and counting True: series.sum() adds its values. For a Boolean mask, True counts as 1 and False as 0. equipment_data.isna().sum() therefore counts missing cells separately for each column.

Finding missing cells: isna() returns True for missing entries such as NaN and None. It returns False for a recorded zero. A DataFrame gives a mask for every cell; a Series gives one mask value per row.

## Training median from the actual ambient readings

*PowerPoint slide 15*

training_ambient_median: median ambient temperature calculated only from training rows.

Run the cell and read the output below it.

In [ ]:
training_ambient_median = training_data["ambient_c"].median()
filled_validation_ambient = validation_data["ambient_c"].fillna(
    training_ambient_median
)
print(round(float(training_ambient_median), 3))
print(validation_data["ambient_c"].isna().sum())
print(filled_validation_ambient.head())

## Preparing X and y, with a filled copy for tree experiments

*PowerPoint slide 16*

X holds the input columns. y holds the answers. The raw X keeps its blanks for pipelines.
median() is worked out from the training rows only.
The filled copies put that same median into any blank, in both tables.

The validation rows borrow the training median and never contribute to it.

In [ ]:
X_train = training_data[feature_columns]
y_train = training_data["target"]
X_valid = validation_data[feature_columns]
y_valid = validation_data["target"]
fill_values = X_train.median()
X_train_filled = X_train.fillna(fill_values)
X_valid_filled = X_valid.fillna(fill_values)
print(X_train.shape, X_valid.shape)

## A flexible model can memorise the training rows

*PowerPoint slide 17*

Same dots, three trees. Depth 1 is too simple. Depth 12 chases every dot, noise included.

## Training error falls to almost zero, validation error stays far above it

*PowerPoint slide 18*

The loop tries three tree depths.
mean_absolute_error(actual, predicted) is the MAE you computed by hand.
We score the same tree twice: on rows it learned from, and on rows it never saw.

Errors are in kWh/t. The deeper tree fits its training rows almost perfectly.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
print("depth train valid")
for depth in [1, 4, 8]:
    tree = DecisionTreeRegressor(
        max_depth=depth, random_state=17)
    tree.fit(X_train_filled, y_train)
    train_pred = tree.predict(X_train_filled)
    valid_pred = tree.predict(X_valid_filled)
    train_mae = mean_absolute_error(y_train, train_pred)
    valid_mae = mean_absolute_error(y_valid, valid_pred)
    print(depth, round(train_mae, 2), round(valid_mae, 2))

**Example output**

```text
depth train valid
1 1.33 1.56
4 0.48 1.2
8 0.03 1.26
```

## The gap between training and validation error is the warning sign

*PowerPoint slide 19*

Depth 1 to 10, computed from this cohort's rows. The orange line is the honest score.

## One validation group can mislead

*PowerPoint slide 20*

THE PROBLEM
Our validation group, E5, has only 15 rows. One odd machine or one lucky stretch of data can make a model look better or worse than it is.

THE REMEDY
Score the model several times, each time holding back a different group. Report the average error and how far the scores spread. That is cross-validation.

## The Week 8 pipeline, ready to be scored on several folds

*PowerPoint slide 22*

This is the Week 8 pipeline: fill blanks with the median, then fit a line.
Nothing is fitted yet. The pipeline is a recipe.
Each fold will fit a fresh copy of this recipe on its own training groups.

A fresh median is learned inside every fold, so no fold borrows from its held-back group.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
linear_model = make_pipeline(
    SimpleImputer(strategy="median"),
    LinearRegression(),
)

## Each fold takes a turn at validation

*PowerPoint slide 23*

An illustration from another engineering field (strength in MPa). Our own fold errors follow on later slides.

## Cross-validation within training data

*PowerPoint slide 24*

GroupKFold keeps each equipment group within one fold. Each fold takes a turn as validation while the remaining folds train the pipeline.

cross_val_score reports one score per fold. neg_mean_absolute_error returns negative MAE, so negate the scores before reporting the fold errors.

## Operations for the next example

*PowerPoint slide 25*

01

A labelled column: pd.Series(values) creates a one-dimensional labelled sequence. With an array of fold errors, the default labels are 0, 1, 2 and so on.

02

Mean of recorded values: series.mean() adds known values and divides by their count. pandas skips missing values by default. The result has the same unit as the original column.

03

Spread of fold errors: series.std(ddof=0) measures how far values spread around their mean. ddof=0 divides squared deviations by the number of values. This describes these folds, not a confidence interval.

## Keeping groups together / Scoring repeated splits

*PowerPoint slide 26*

GroupKFold(n_splits=4) creates four train/validation splits. All rows of an equipment group stay on one side of each split. Four splits require at least four distinct training groups.

cross_val_score fits a fresh model for each split and scores its held-out rows. groups identifies the equipment groups and cv supplies the split rule. neg_mean_absolute_error returns negative MAE; negate it to report positive errors.

## Repeat evaluation without mixing groups

*PowerPoint slide 27*

Four folds means four held-out group turns.

In each turn, learn a fresh pipeline on the other training groups and score the held-out group.

This uses training_data only; validation and test tables stay outside this calculation.

## Fold errors calculated from the supplied training rows: step 1

*PowerPoint slide 28*

grouped_folds: the split rule that keeps observations from one equipment group together.

Run the cell and read the output below it.

In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_score
grouped_folds = GroupKFold(n_splits=4)

## Read the scoring convention

*PowerPoint slide 29*

scikit-learn uses larger scores as better. neg_mean_absolute_error therefore returns negative errors.

Negate the returned scores to recover ordinary positive MAE; lower MAE is better.

## Fold errors calculated from the supplied training rows: step 2

*PowerPoint slide 30*

grouped_folds: the split rule that keeps observations from one equipment group together.

Run the cell and read the output below it.

In [ ]:
negative_error_scores = cross_val_score(
    linear_model,
    X_train,
    y_train,
    groups=training_data["equipment_id"],
    cv=grouped_folds,
    scoring="neg_mean_absolute_error",
)

## Report typical error and variation

*PowerPoint slide 31*

Each value is the MAE of one held-out fold. mean gives average error. std(ddof=0) describes the spread across these four errors only.

A large spread suggests performance depends on the held-out equipment.

## Fold errors calculated from the supplied training rows: step 3

*PowerPoint slide 32*

Run the cell and read the output below it.

Run the cell and read the output below it.

In [ ]:
fold_absolute_errors = pd.Series(-negative_error_scores)
print(fold_absolute_errors.round(4).tolist())
print(round(float(fold_absolute_errors.mean()), 4))
print(round(float(fold_absolute_errors.std(ddof=0)), 4))

## Leakage makes a score look better than it is

*PowerPoint slide 33*

Data leakage: information the model should not have reaches the model during training.

Think of an exam. If the exam paper is on your desk the night before, a high mark proves nothing about what you learned.
A leaky score is a mark earned with the paper in hand. It looks wonderful and it fails in real use.
Leakage is common and quiet: the code runs, the score is high, and nothing warns you.

## Four ways leakage happens, and a guard for each

*PowerPoint slide 34*

Each guard is a habit you can apply while preparing the data.

## A leaky score looks wonderful

*PowerPoint slide 35*

concat stacks two tables on top of each other.
The first score is honest: the tree never saw the validation rows.
The second score is leaky: we put the validation rows into training first.

Both scores are validation errors in kWh/t. Only the second one is invalid.

In [ ]:
import pandas as pd
leaky_X = pd.concat([X_train_filled, X_valid_filled])
leaky_y = pd.concat([y_train, y_valid])
tree = DecisionTreeRegressor(
    max_depth=10, random_state=17)
tree.fit(X_train_filled, y_train)
pred = tree.predict(X_valid_filled)
print("honest", round(mean_absolute_error(y_valid, pred), 2))
tree.fit(leaky_X, leaky_y)
pred = tree.predict(X_valid_filled)
print("leaky ", round(mean_absolute_error(y_valid, pred), 2))

**Example output**

```text
honest 1.18
leaky  0.01
```

## Pandas inside the required function

*PowerPoint slide 36*

The assignment caller supplies equipment_records. pd.DataFrame(records) gives those records named columns. Calculate with pandas, then return the published Python type.

to_dict("records") makes a list of row dictionaries. Series.tolist() makes a list of values. A grouped Series.to_dict() maps group labels to validation_results.

## Operations for the next example

*PowerPoint slide 37*

01

Returning a dictionary: series.to_dict() maps Series labels to values. equipment_data.to_dict("records") instead returns a list of row dictionaries. Choose the form required by the assignment.

02

A table from Python values: pd.DataFrame(records) builds a table from row dictionaries. With a dictionary of column lists, keys become column names and each list supplies that column’s values.

03

Grouping rows: equipment_data.groupby("label")["value"] groups the value column by matching labels. Apply mean() or sum() to calculate one result per group. Missing group labels are excluded by default.

## Defining and calling a function

*PowerPoint slide 38*

def gives a reusable block a name and input parameters. The indented statements run when it is called. return sends a result back to the caller; print displays a result but does not return it.

## A function called with the supplied records

*PowerPoint slide 39*

equipment_records: the input list of row dictionaries supplied to the function.

Run the cell and read the output below it.

In [ ]:
def mean_load_by_equipment(equipment_records):
    equipment_data = pd.DataFrame(equipment_records)
    mean_load_by_equipment_group = equipment_data.groupby(
        "equipment_id"
    )["load_pct"].mean()
    return mean_load_by_equipment_group.to_dict()
equipment_records = modelling_data.to_dict("records")
print(mean_load_by_equipment(equipment_records))

## Block 1, part C: this week's tasks

*PowerPoint slide 41*

Part C of 3. 20 marks.

Today's tasks are part C of the Block 1 pack, worth 20 of its 60 marks. They use group_split and predict_regression.
This is the last part. The pack closes Friday 20 November 2026, 08:00 Kampala time. Part A and part B are the earlier weeks' notebooks.
The quiz opens after today's class.

## Your issued task

*PowerPoint slide 42*

01

Investigate crusher energy demand.

02

Target units: kWh/t.

03

These are synthetic teaching records for practice.

04

Each record is one operating observation; equipment_id repeats across observations.

## group_split(records)

*PowerPoint slide 43*

Assign whole equipment groups: E1–E4 to train, E5 to validation, E6 to test.

Return train, validation and test lists of record_id in input order.

Do not use target values to choose membership.

Keep the exact names and outputs. Check types, units and the issued brief.

## predict_regression(training_records, feature_records)

*PowerPoint slide 44*

Fit a training-mean baseline, linear regression and a decision tree (max_depth=4, random_state=17) using load_pct and ambient_c only.

Return three prediction lists under baseline, linear and tree.

Feature records have no target.

Fit preprocessing only on training data.

Keep the exact names and outputs. Check types, units and the issued brief.

## Written evidence

*PowerPoint slide 45*

01

Use the cell headed “My written evidence”.

02

Explain the supplied scenario in your own words.

03

State target units, reasons and limitations.

04

Submit one complete notebook containing code and written evidence.

## Submit the Block 1 assignment pack

*PowerPoint slide 46*

01

Part A: Regression and baseline (A7, Week 8)

Open the A7 form

02

Part B: Classification and cost (A8, Week 9)

Open the A8 form

03

Part C: Validation and trust (A9, Week 10)

Open the A9 form

Closes Friday 20 November 2026, 08:00 Kampala time. Submit each notebook to its own form, with your full Student Registration Number.